# Platform, Community, And Web Workflows

Examples for platform deployment plans, infrastructure delivery, identity and enterprise APIs, operations/AI/HA flows, Community social workflows, and web rendering surfaces.

Run this cell from the repository root after `npm ci` and `npm run build`. The stored output below was regenerated by `npm run notebooks:build`.

## Platform Deployment, Review, Package, Search, And Readiness

**Use when:** Use this flow when headless automation needs to create a deployable service and prove production readiness.

The next cell is the executable example.

In [1]:
import { createInMemoryPlatformCore } from 'epoch/Epoch.Platform.Core';
import { EpochPlatformSdk } from 'epoch/Epoch.Platform.Sdk';

const sdk = new EpochPlatformSdk(createInMemoryPlatformCore({ communityEnabled: false }));
const organization = sdk.organizations.create({ slug: 'acme', displayName: 'Acme' });
const project = sdk.projects.create({ organizationId: organization.id, slug: 'platform', displayName: 'Platform' });
const repository = sdk.repositories.create({ projectId: project.id, slug: 'api', visibility: 'private' });
const environment = sdk.environments.create({ projectId: project.id, name: 'production', type: 'production', protected: true });
const deployable = sdk.deployables.create({ projectId: project.id, name: 'api-web', kind: 'app', source: { repositoryId: repository.id } });
const runner = sdk.runners.register({ name: 'runner-1', capacity: 2 });
sdk.runners.heartbeat({ name: runner.name });
sdk.backups.configureDestination({ uri: 'file://backups' });
const serviceAccount = sdk.identity.createServiceAccount({ organizationId: organization.id, name: 'deployer', scopes: ['deployments:write'] });
const secret = sdk.secrets.create({ environmentId: environment.id, name: 'DATABASE_URL', value: 'postgres://secret' });
sdk.secrets.grantAccess(secret.id, { serviceAccountId: serviceAccount.id });
const plan = sdk.deployments.createPlan({ deployableId: deployable.id, environmentId: environment.id });
sdk.deployments.approvePlan(plan.id, { actor: 'ops-lead' });
const deployment = sdk.deployments.executePlan(plan.id);
const issue = sdk.issues.create({ projectId: project.id, title: 'Document deploy' });
const review = sdk.reviews.createIntent({ repositoryId: repository.id, issueId: issue.id, title: 'Update deploy docs' });
sdk.reviews.recordCheck(review.id, { name: 'unit', status: 'passed' });
sdk.reviews.approve(review.id, { actor: 'bob' });
const mergedReview = sdk.reviews.merge(review.id);
const pkg = sdk.packages.publish({ name: 'api-web', version: '1.0.0', deploymentId: deployment.id });
const dashboard = sdk.operations.dashboard();
const readiness = sdk.operations.firstRunReadiness();

console.log(JSON.stringify({
  organization: organization.slug,
  project: project.slug,
  deploymentState: deployment.state,
  jobState: sdk.jobs.get(deployment.jobId).state,
  issueTitle: issue.title,
  reviewState: mergedReview.state,
  packageVersion: pkg.version,
  searchLabels: sdk.search.query('api').map((result) => result.label),
  productionReady: readiness.productionReady,
  serviceHealth: dashboard.serviceHealth,
  runnerCapacity: dashboard.runnerCapacity,
}, null, 2));

{
  "organization": "acme",
  "project": "platform",
  "deploymentState": "succeeded",
  "jobState": "succeeded",
  "issueTitle": "Document deploy",
  "reviewState": "merged",
  "packageVersion": "1.0.0",
  "searchLabels": [
    "api",
    "api-web",
    "api-web"
  ],
  "productionReady": true,
  "serviceHealth": "healthy",
  "runnerCapacity": 2
}


**How to read the output:** The platform SDK creates the deploy graph, protects execution with approval, records a successful job, publishes a package, and reports readiness once runners, backups, and deployment exist.

## Infrastructure Delivery, Jobs, And Configuration Validation

**Use when:** Use this flow when operators need inspectable infrastructure targets, dry-run plans, retryable jobs, and forward-compatible config loading.

The next cell is the executable example.

In [2]:
import { createInMemoryPlatformCore, PlatformError } from 'epoch/Epoch.Platform.Core';
import { EpochPlatformSdk } from 'epoch/Epoch.Platform.Sdk';

const sdk = new EpochPlatformSdk(createInMemoryPlatformCore());
const org = sdk.organizations.create({ slug: 'acme', displayName: 'Acme' });
const project = sdk.projects.create({ organizationId: org.id, slug: 'api', displayName: 'API' });
const repo = sdk.repositories.create({ projectId: project.id, slug: 'api', visibility: 'private' });
const env = sdk.environments.create({ projectId: project.id, name: 'staging', type: 'preview', protected: false });
const target = sdk.infrastructure.connectTarget({ organizationId: org.id, name: 'primary', kind: 'kubernetes', region: 'us-central1' });
const resource = sdk.resources.provision({ projectId: project.id, environmentId: env.id, name: 'db', kind: 'postgres', provider: 'neon' });
sdk.templates.create({ name: 'node-web', kind: 'app' });
const deployable = sdk.deployables.discover({ projectId: project.id, repositoryId: repo.id, name: 'api-web', kind: 'app', runtime: 'node', manifestPath: 'package.json' });
const dryRun = sdk.deployments.createPlan({ deployableId: deployable.id, environmentId: env.id, dryRun: true, idempotencyKey: 'plan-1' });
const edited = sdk.deployments.editPlan(dryRun.id, { runtimeVariables: { NODE_ENV: 'staging' } });
const canceled = sdk.deployments.cancelPlan(edited.id, { actor: 'ops' });
const plan = sdk.deployments.createPlan({ deployableId: deployable.id, environmentId: env.id, idempotencyKey: 'plan-2' });
sdk.deployments.approvePlan(plan.id, { actor: 'ops' });
const deployment = sdk.deployments.executePlan(plan.id);
const promoted = sdk.deployments.promote(deployment.id, { environmentId: env.id });
const job = sdk.jobs.schedule({ name: 'reconcile', type: 'runner-reconcile', idempotencyKey: 'job-1' });
const scheduledState = job.state;
const retried = sdk.jobs.retry(job.id);
const retryAttempt = retried.attempt;
const reconciled = sdk.jobs.reconcileRunnerLoss(job.id);
const validConfig = sdk.configuration.load({ unknownFutureSection: { keep: true }, runners: { capacity: 1 } });

let invalidConfigCode = '';
try {
  sdk.configuration.load({ database: { invalid: true } });
} catch (error) {
  invalidConfigCode = error instanceof PlatformError ? error.code : 'unknown';
}

console.log(JSON.stringify({
  targetKind: target.kind,
  resourceProvider: resource.provider,
  templates: sdk.templates.list().map((item) => item.name),
  discoveredRuntime: sdk.deployables.get(deployable.id).runtime,
  canceledState: canceled.state,
  promotionEnvironmentId: promoted.environmentId,
  jobLifecycle: { scheduledState, retryAttempt, reconciledState: reconciled.state },
  configWarnings: validConfig.warnings,
  invalidConfigCode,
}, null, 2));

{
  "targetKind": "kubernetes",
  "resourceProvider": "neon",
  "templates": [
    "node-web"
  ],
  "discoveredRuntime": "node",
  "canceledState": "canceled",
  "promotionEnvironmentId": "env_10",
  "jobLifecycle": {
    "scheduledState": "scheduled",
    "retryAttempt": 2,
    "reconciledState": "reconciled"
  },
  "configWarnings": [
    "unknownFutureSection"
  ],
  "invalidConfigCode": "invalid_configuration"
}


**How to read the output:** Unknown future config sections produce warnings, invalid known sections fail with typed errors, and platform jobs retain auditable retry/reconcile state.

## Enterprise Identity, API Correlation, Webhooks, Compliance, And Tenant Export

**Use when:** Use this flow when enterprise control-plane automation needs auditable identity and tenant lifecycle APIs.

The next cell is the executable example.

In [3]:
import { createInMemoryPlatformCore, signWebhookPayload } from 'epoch/Epoch.Platform.Core';
import { EpochPlatformSdk, PlatformError } from 'epoch/Epoch.Platform.Sdk';

const sdk = new EpochPlatformSdk(createInMemoryPlatformCore());
const org = sdk.organizations.create({ slug: 'acme', displayName: 'Acme' });
const user = sdk.identity.createUser({ handle: 'alice', displayName: 'Alice' });
const sso = sdk.identity.configureSsoProvider({ name: 'Okta', protocol: 'oidc' });
const scimUser = sdk.identity.provisionScimUser({ handle: 'bob', group: 'engineering' });
const serviceAccount = sdk.identity.createServiceAccount({ organizationId: org.id, name: 'automation', scopes: ['projects:write'] });
const token = sdk.identity.issueApiToken({ serviceAccountId: serviceAccount.id, name: 'ci' });
const session = sdk.identity.openSession({ userId: user.id, name: 'browser' });
sdk.api.openRequest({ correlationId: 'req-1', idempotencyKey: 'deploy-plan-1' });
const completed = sdk.api.completeRequest('req-1');
sdk.api.registerWebhookEndpoint({ name: 'ops', secret: 'whsec' });
const payload = JSON.stringify({ event: 'deployment.executed' });
const signature = signWebhookPayload('whsec', payload);
const webhookOk = sdk.api.verifyWebhook({ endpointName: 'ops', payload, signature });
sdk.compliance.setAuditRetention({ retention: '7y' });
const report = sdk.compliance.report();
sdk.projects.create({ organizationId: org.id, slug: 'api', displayName: 'API' });
const tenantExport = sdk.tenants.export({ organizationId: org.id });
const deletedExport = sdk.tenants.deleteExport({ organizationSlug: tenantExport.slug });
sdk.identity.revokeApiToken(token.id);
sdk.identity.revokeSession(session.id);

let typedError = {};
try {
  sdk.api.sampleError({ code: 'invalid_request' });
} catch (error) {
  if (error instanceof PlatformError) typedError = { code: error.code, hasAuditEvent: error.auditEventId !== undefined };
}

console.log(JSON.stringify({
  ssoName: sso.name,
  scimHandle: scimUser.handle,
  tokenState: sdk.identity.getApiToken(token.id).state,
  sessionState: sdk.identity.getSession(session.id).state,
  completedCorrelation: completed.correlationId,
  webhookOk,
  complianceFindings: report.findings,
  tenantProjects: tenantExport.projectSlugs,
  tenantExportDeleted: deletedExport.deleted,
  auditEvents: sdk.compliance.exportAudit().length,
  typedError,
}, null, 2));

{
  "ssoName": "Okta",
  "scimHandle": "bob",
  "tokenState": "revoked",
  "sessionState": "revoked",
  "completedCorrelation": "req-1",
  "webhookOk": true,
  "complianceFindings": [
    "encrypted secrets at rest",
    "audit retention 7y"
  ],
  "tenantProjects": [
    "api"
  ],
  "tenantExportDeleted": true,
  "auditEvents": 14,
  "typedError": {
    "code": "invalid_request",
    "hasAuditEvent": false
  }
}


**How to read the output:** The enterprise APIs keep identity state explicit, verify webhook signatures, expose compliance findings, export/delete tenant data, and return typed diagnostics.

## Operations, AI Guardrails, Backup, Restore, And HA Drills

**Use when:** Use this flow when production operators need incident response, AI context without secret leakage, backup verification, and failover proof.

The next cell is the executable example.

In [4]:
import { createInMemoryPlatformCore } from 'epoch/Epoch.Platform.Core';
import { EpochPlatformSdk, PlatformError } from 'epoch/Epoch.Platform.Sdk';

const sdk = new EpochPlatformSdk(createInMemoryPlatformCore({ communityEnabled: true }));
const organization = sdk.organizations.create({ slug: 'acme', displayName: 'Acme' });
const project = sdk.projects.create({ organizationId: organization.id, slug: 'platform', displayName: 'Platform' });
const repository = sdk.repositories.create({ projectId: project.id, slug: 'api', visibility: 'private' });
const environment = sdk.environments.create({ projectId: project.id, name: 'production', type: 'production', protected: true });
const deployable = sdk.deployables.create({ projectId: project.id, name: 'api-web', kind: 'app', source: { repositoryId: repository.id } });
sdk.runners.register({ name: 'runner-1', capacity: 1 });
sdk.backups.configureDestination({ uri: 'file://backups' });
const plan = sdk.deployments.createPlan({ deployableId: deployable.id, environmentId: environment.id });
sdk.deployments.approvePlan(plan.id, { actor: 'ops' });
const deployment = sdk.deployments.executePlan(plan.id);
const incident = sdk.incidents.markDeploymentFailed(deployment.id, { failureClass: 'health-check' });
sdk.incidents.acknowledge(incident.id, { actor: 'oncall' });
const followUp = sdk.incidents.createFollowUpIssue(incident.id, { title: 'Fix health check' });
const rollback = sdk.deployments.rollback(deployment.id, { actor: 'oncall' });
const aiContext = sdk.ai.createContextPack({ actor: 'oncall', projectId: project.id, sources: ['logs', 'checks', 'secrets'] });
const deployTool = sdk.ai.requestTool({ tool: 'deploy.rollback', projectId: project.id });

let deniedToolCode = '';
try {
  sdk.ai.requestTool({ tool: 'secret.reveal', projectId: project.id });
} catch (error) {
  deniedToolCode = error instanceof PlatformError ? error.code : 'unknown';
}

const backup = sdk.backups.start({ name: 'nightly' });
const verified = sdk.backups.verify(backup.id);
const restore = sdk.restores.dryRun({ backupId: backup.id });
const ha = sdk.ha.declareProfile({ name: 'standard', rpo: '15m', rto: '1h' });
const drill = sdk.ha.runFailoverDrill({ name: 'quarterly' });

console.log(JSON.stringify({
  incidentClass: incident.failure?.classification,
  followUpTitle: followUp.title,
  rollbackState: rollback.state,
  aiContextBody: aiContext.body,
  aiContextCitations: aiContext.citations,
  deployToolRequiresApproval: deployTool.requiresApproval,
  deniedToolCode,
  backupVerification: verified.verificationStatus,
  restoreStatus: restore.status,
  haProfile: { name: ha.name, rpo: ha.rpo, rto: ha.rto },
  drillStatus: drill.status,
}, null, 2));

{
  "incidentClass": "health_check",
  "followUpTitle": "Fix health check",
  "rollbackState": "rolled_back",
  "aiContextBody": "AI context for oncall: logs, checks",
  "aiContextCitations": [
    "logs",
    "checks"
  ],
  "deployToolRequiresApproval": true,
  "deniedToolCode": "policy_denied",
  "backupVerification": "verified",
  "restoreStatus": "passed",
  "haProfile": {
    "name": "standard",
    "rpo": "15m",
    "rto": "1h"
  },
  "drillStatus": "passed"
}


**How to read the output:** The AI context excludes secrets from citations/body, sensitive tools are denied, deploy tools require approval, and operations artifacts show rollback, backup verification, restore dry-run, and HA drill status.

## Community Workflows And Browser Web Surfaces

**Use when:** Use this flow when Community collaboration and web console rendering should be verified without a live server.

The next cell is the executable example.

In [5]:
import { JSDOM } from 'jsdom';
import { createInMemoryPlatformCore } from 'epoch/Epoch.Platform.Core';
import { EpochPlatformSdk } from 'epoch/Epoch.Platform.Sdk';
import { createInMemoryCommunityApi } from 'epoch/Epoch.Community.API';
import { createCommunityClient } from 'epoch/Epoch.Community.Core';
import { createCommunityWebApp, renderCommunityWebDocument } from 'epoch/Epoch.Community.Web';
import { createPlatformWebApp, renderPlatformConsole } from 'epoch/Epoch.Platform.Web';

const sdk = new EpochPlatformSdk(createInMemoryPlatformCore({ communityEnabled: false }));
const org = sdk.organizations.create({ slug: 'acme', displayName: 'Acme' });
const project = sdk.projects.create({ organizationId: org.id, slug: 'api', displayName: 'API' });
sdk.community.enable({ reviewedBy: 'admin', visibilityPolicy: 'approved project showcases' });
const profile = sdk.community.createProfile({ handle: 'alice', displayName: 'Alice', visibility: 'public' });
const showcase = sdk.community.publishProject({
  projectId: project.id,
  publicSlug: 'api-web',
  summary: 'Self-hosted API platform',
  requestedBy: 'alice',
  readme: 'Deployable API service.',
  deployStatusBadge: 'healthy',
  contributionPrompt: 'Start with starter issues.',
});
const approved = sdk.community.approveProject(showcase.id, { moderator: 'mod' });
sdk.community.addTopic(approved.id, { topic: 'self-hosting' });
sdk.community.addShowcaseAsset(approved.id, { asset: 'sha256:demo-screenshot' });
sdk.community.publishRelease(approved.id, { version: '1.0.0' });
sdk.community.followProject({ profileId: profile.id, communityProjectId: approved.id });
sdk.community.starProject({ profileId: profile.id, communityProjectId: approved.id });
sdk.community.bookmarkProject({ profileId: profile.id, communityProjectId: approved.id });
const discussion = sdk.community.openDiscussion({ communityProjectId: approved.id, profileId: profile.id, title: 'Roadmap', body: 'Where should deploy previews land?' });
const report = sdk.community.reportDiscussion({ discussionId: discussion.id, reason: 'needs moderation' });
sdk.community.resolveReport(report.id, { moderator: 'mod' });
const legalHold = sdk.community.exportLegalHold({ communityProjectId: approved.id });

const client = createCommunityClient(createInMemoryCommunityApi());
await client.createRepository({ slug: 'api-web', displayName: 'API Web', description: 'Self-hosted API platform', maintainers: ['alice'] });
await client.openIssue('api-web', { title: 'Add release docs', author: 'bob' });
await client.proposeChange('api-web', { title: 'Update docs', author: 'alice', sourceView: 'feature/docs', targetView: 'main' });
await client.reviewChange('api-web', 'CHANGE-1', { reviewer: 'maintainer', decision: 'approved' });
const appDefinition = await createCommunityWebApp({ client, basePath: '/community' });
const communityDocument = renderCommunityWebDocument(appDefinition);

const dom = new JSDOM('<div id="root"></div>');
globalThis.window = dom.window;
const root = dom.window.document.getElementById('root');
renderPlatformConsole(root, {
  role: 'operator',
  productionReady: true,
  projectName: 'platform',
  environmentName: 'production',
  deployableName: 'api-web',
  primaryAction: 'Deploy to production',
  deploymentHealth: 'healthy',
  communityEnabled: true,
  runnerCount: 1,
  latestDeploymentState: 'succeeded',
  packageName: 'api-web',
  packageVersion: '1.0.0',
  searchResults: [{ type: 'repository', label: 'api' }],
  mobileActions: ['Approve', 'Rollback', 'Ask AI'],
  homeModules: ['pending reviews', 'risky deploys'],
  adminSections: ['identity and SSO', 'upgrade and support bundle'],
  sdkEquivalent: 'sdk.deployments.executePlan(plan.id)',
});

console.log(JSON.stringify({
  communityEnabled: sdk.community.status().enabled,
  profileHandle: profile.handle,
  projectState: approved.moderationState,
  topics: sdk.community.getProjectBySlug('api-web').topics,
  feedVerbs: sdk.community.feed().map((event) => event.verb),
  personalizedFeedCount: sdk.community.personalizedFeed({ profileId: profile.id }).length,
  moderationQueueCount: sdk.community.moderationQueue().length,
  legalHoldDiscussions: legalHold.discussions.length,
  platformAppRoutes: createPlatformWebApp().routes.map((route) => route.id),
  renderedTextIncludesAction: root.textContent.includes('Deploy to production'),
  renderedTextIncludesSdk: root.textContent.includes('sdk.deployments.executePlan'),
  communityRoutes: appDefinition.routes.map((route) => route.path),
  communityRepositorySummary: appDefinition.repositories.map((repo) => ({
    slug: repo.slug,
    issues: repo.issues.length,
    proposals: repo.changeProposals.length,
    proposalStatus: repo.changeProposals[0]?.status,
  })),
  communityDocumentIncludesSlug: communityDocument.includes('api-web'),
}, null, 2));

{
  "communityEnabled": true,
  "profileHandle": "alice",
  "projectState": "approved",
  "topics": [
    "self-hosting"
  ],
  "feedVerbs": [
    "release.published",
    "discussion.opened"
  ],
  "personalizedFeedCount": 2,
  "moderationQueueCount": 0,
  "legalHoldDiscussions": 1,
  "platformAppRoutes": [
    "services",
    "deployments",
    "backups",
    "health",
    "settings"
  ],
  "renderedTextIncludesAction": true,
  "renderedTextIncludesSdk": true,
  "communityRoutes": [
    "/community/repository-browsing",
    "/community/issue-tracking",
    "/community/change-review",
    "/community/discussion-threads",
    "/community/maintainer-profiles",
    "/community/release-discovery",
    "/community/organization-spaces"
  ],
  "communityRepositorySummary": [
    {
      "slug": "api-web",
      "issues": 1,
      "proposals": 1,
      "proposalStatus": "approved"
    }
  ],
  "communityDocumentIncludesSlug": true
}


**How to read the output:** The result covers Community moderation and social state, the standalone Community web definition, and DOM-rendered Platform console output without needing a browser server.